In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

In [ ]:
# Ruta a las carpetas de espectrogramas
DATA_DIR = "../data/spectrograms"
BATCH_SIZE = 32
IMG_SIZE = (969, 370)

# 1. Cargar dataset desde carpetas organizadas por clase
raw_train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

raw_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# ✅ Obtener nombres de clases antes de transformar los datasets
class_names = raw_train_ds.class_names
num_classes = len(class_names)
print("Clases detectadas:", class_names)

# 2. Optimización de dataset
AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = raw_val_ds.cache().prefetch(buffer_size=AUTOTUNE)


# 3. Crear modelo CNN simple
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes)
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# 4. Entrenar el modelo
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# 5. Guardar modelo
model.save("modelo_espectrogramas.keras")

# 6. Graficar resultados
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.legend()
plt.title("Accuracy por época")
plt.show()


Found 999 files belonging to 10 classes.
Using 800 files for training.
Found 999 files belonging to 10 classes.
Using 199 files for validation.
Clases detectadas: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Epoch 1/10
 5/25 ━━━━━━━━━━━━━━━━━━━━ 3:19 10s/step - accuracy: 0.1173 - loss: 16.9445